In [ ]:
import torch
import torch.nn as nn
import numpy as np
import wandb
from torch.utils.data import DataLoader, Dataset
import os
from torchvision import transforms
from transformers import AutoModel, AutoTokenizer
from diffusers import AutoencoderDC
import matplotlib.pyplot as plt
import random
from PIL import Image
from torchvision.models import vgg16
import pandas as pd
import torch.nn.functional as Fn
from einops import rearrange
from torch.optim.lr_scheduler import StepLR
from tqdm import tqdm
from torchviz import make_dot
import os

if(torch.cuda.is_available()):
    device = torch.device('cuda')
else:
    device = torch.device('mps')

torch.autograd.set_detect_anomaly(True)
print("Device: ", device)

Device:  mps


In [2]:
modelPath = "./models/"
os.makedirs(modelPath, exist_ok=True)
QWENDIR = os.path.join(modelPath, "qwen3-embedding-8b")
AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-8B").save_pretrained(QWENDIR)
AutoModel.from_pretrained("Qwen/Qwen3-Embedding-8B").save_pretrained(QWENDIR)
QWEN3TOKENIZER = AutoTokenizer.from_pretrained(QWENDIR, local_files_only=True)
QWEN3MODEL = AutoModel.from_pretrained(QWENDIR, local_files_only=True)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

In [3]:
DCAEENCODER = AutoencoderDC.from_pretrained(f"mit-han-lab/dc-ae-f64c128-in-1.0-diffusers", torch_dtype=torch.float32).to(device).eval()

def concatenateTextEmbeddings(text, maxLength = 512, modelPath = "./models"):
    if isinstance(text, str):
        text = [text]
    elif isinstance(text, (list, tuple)):
        pass
    else:
        raise ValueError(f"Give string or list of strings, recieved this {type(text)}")
    
    input3 = QWEN3TOKENIZER(text, padding='max_length', return_tensors="pt", truncation=True, max_length=maxLength)

    with torch.no_grad():
        output3 = QWEN3MODEL(**input3)
        embeddings3 = output3.last_hidden_state
    
    textEmbeddings = embeddings3
    return textEmbeddings


In [ ]:
def precompute_embeddings_and_latents(data_path, root_dir, output_dir, device):
    """
    Pre-computes and saves image latents and text embeddings.
    """
    print(f"Starting pre-computation on device: {device}")
    
    
    os.makedirs(output_dir, exist_ok=True)
    
    data = pd.read_csv(data_path)
    total_samples = len(data)
    
    preprocess_vae = transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])
    
    for index, row in tqdm(data.iterrows(), total=total_samples, desc="Pre-computing"):
        image_path = os.path.join(root_dir, row['imagePath'])
        
        base_name = os.path.splitext(os.path.basename(image_path))[0]
        output_file = os.path.join(output_dir, f"{base_name}.pt")
        
        if os.path.exists(output_file):
            continue # Skip if already processed

        try:
            image = Image.open(image_path).convert("RGB")
            image_tensor = preprocess_vae(image).unsqueeze(0).to(device) 
            
            with torch.no_grad():
                latent = DCAEENCODER.encode(image_tensor).latent
                
        except Exception as e:
            print(f"Skipping image {image_path} due to error: {e}")
            continue

        captions = [
            row['caption1'], row['caption2'], row['caption3'],
            row['caption4'], row['caption5']
        ]
        tembed1 = concatenateTextEmbeddings(captions[0]).squeeze(0).cpu() # [512, 4096]
        tembed2 = concatenateTextEmbeddings(captions[1]).squeeze(0).cpu() # [512, 4096]
        tembed3 = concatenateTextEmbeddings(captions[2]).squeeze(0).cpu() # [512, 4096]
        tembed4 = concatenateTextEmbeddings(captions[3]).squeeze(0).cpu() # [512, 4096]
        tembed5 = concatenateTextEmbeddings(captions[4]).squeeze(0).cpu() # [512, 4096]


        torch.save({
            'latent': latent.squeeze(0).cpu(), # [128, 8, 8]
            'tembed1': tembed1,
            'tembed2': tembed2,
            'tembed3': tembed3,
            'tembed4': tembed4,
            'tembed5': tembed5,
        }, output_file)

    print("Pre-computation complete.")
    
latentDir = "/Users/ishananand/Desktop/Tiny-Recursive-Model-for-Text-To-Image-Generation/precomputedLatents" 
precompute_embeddings_and_latents("dataset/COCO2017.csv", "./", latentDir, device) 

Starting pre-computation on device: mps


Pre-computing:   0%|          | 1/123287 [06:04<12470:19:56, 364.14s/it]


KeyboardInterrupt: 